# PHYS 771 Assignment 1 due June 17 2026 11:59pm

The goal of this exercise will be to calculate the radial velocity curve of a planetary system discovered by the Kepler satellite (Crossfield et al. 2015; http://arxiv.org/abs/ 1501.03798). You will study the feasibility of measuring the masses of these planets with a high-resolution spectrometer that reaches the meter per second precision. These three Super-Earth planets were discovered around an M-type low-mass star (named K2−3) with the transit method. The physical parameters of the planets are given in Table 2 of the paper. For this exercise, we will suppose that all three planets have co-planar and circular orbits, with an inclination of exactly 90 degrees. We will also suppose that the impact parameter of the transit is null and that all arguments of pericentre are equal at 90 degrees.

#### 0. Imports

In [102]:
import astropy.units as u
import astropy.constants as const
import numpy as np

### a) Determine the masses of the three planets, in units of Earth masses. Assume that K2−3 b and K2−3 c have volumetric densities equal to that of Jupiter (1.33 g/cm3) and that K2−3 d has a volumetric density equal to that of the Earth (5.5 g/cm3).

In [103]:
# volumetric densities assumptions 
rho_K2b = 1.33 * u.g / u.cm**3
rho_K2c = 1.33 * u.g / u.cm**3
rho_K2d = 5.5 * u.g / u.cm**3

In [104]:
# planet radii from Table 2 of the paper
R_K2b = 2.14 * u.earthRad
R_K2c = 1.72 * u.earthRad
R_K2d = 1.52 * u.earthRad

In [105]:
# Mass-radius relations from section 2.6 of the paper 
def MassRadius(radius, density):
    """ 
    radius: Rearth 
    density: g/cm**3
    """
    if radius < 1.5 * u.earthRad:
        mass = ((4*np.pi)/3 ) * radius**3 * density 

        if (mass.cgs.unit != u.g):
            print("Units error")
            return None

        return mass

    if radius >= 1.5 * u.earthRad and radius < 4.0 * u.earthRad:
        mass = (2.69 * u.earthMass) * (radius.value)**0.93

        if (mass.cgs.unit != u.g):
            print("Units error")
            return None

        return mass

    if radius >= 4.0 * u.earthRad:
        mass = (1.0 * u.earthMass) * (radius.value)**2.06 

        if (mass.cgs.unit != u.g):
            print("Units error")
            return None

        return mass 

    print('Mistakes have been made.')
    return None

In [106]:
# compute masses from the relations above
M_K2b = MassRadius(R_K2b,rho_K2b)
M_K2c = MassRadius(R_K2c,rho_K2c)
M_K2d = MassRadius(R_K2d,rho_K2d)

In [107]:
print(f"Mass of K2-3b: {M_K2b:.3f}")
print(f"Mass of K2-3c: {M_K2c:.3f}")
print(f"Mass of K2-3d: {M_K2d:.3f}")

Mass of K2-3b: 5.458 earthMass
Mass of K2-3c: 4.454 earthMass
Mass of K2-3d: 3.971 earthMass


### b) Determine the expected radial velocity semi-amplitude (K) for each planet.

In [108]:
# star mass from Table 1 
Mstar = 0.601 * u.solMass

# inclination and eccentricities are given assumptions 
i = 90.0 
e = 0.0

# planet periods from Table 2
P_K2b = 10.05403 * u.day
P_K2c = 24.6454 * u.day
P_K2d = 44.5631 * u.day

In [109]:
# We use the formula from the RV lecture notes (2.26)
def semiAmplitude(period, Mp, Mstar, i, e):
    """ 
    Period: days
    Mp: earthMass 
    Mstar: solMass
    i: unitless float
    e: unitless float
    """ 

    K = ((2*np.pi*const.G)/period)**(1/3) *  (Mp * np.sin(np.deg2rad(i))/((Mstar+Mp)**(2/3)) ) * ( 1/((1-e**2)**(1/2)) )
    
    if (K.cgs.unit != (u.cm/u.s)):
        print("Units error")
        return None

    return K.to(u.m/u.s)

In [110]:
# compute RV semi-amplitudes using the formula 
K_K2b = semiAmplitude(P_K2b, M_K2b, Mstar, i, e)
K_K2c = semiAmplitude(P_K2c, M_K2c, Mstar, i, e)
K_K2d = semiAmplitude(P_K2d, M_K2d, Mstar, i, e)

In [111]:
print(f"RV Semi-amplitude of K2-3b: {K_K2b:.3f}")
print(f"RV Semi-amplitude of K2-3c: {K_K2c:.3f}")
print(f"RV Semi-amplitude of K2-3d: {K_K2d:.3f}")

RV Semi-amplitude of K2-3b: 2.271 m / s
RV Semi-amplitude of K2-3c: 1.374 m / s
RV Semi-amplitude of K2-3d: 1.006 m / s


### c) Determine the radial velocity of the star K2−3 with respect to an observer on Earth,in meters per second at moment t = T0c, where T0c is the epoch of central transit ofthe planet K2−3 c (given in Table 2 of the paper). You can ignore the radial velocity of the center of mass of the K2−3 system, and the Earth’s motion.

In [112]:
# epoch of central transit of each planet from Table 2
T_0b = 1980.4189 * u.day
T_0c = 1979.2786 * u.day
T_0d = 1993.2232 * u.day

# ignored COM motion and Earth motion
v_sys = 0.0 * (u.m /u.s)

# given that the argument of pericenter is 90 deg
w = 90.0

In [118]:
# since we are assuming circular orbits, the true anomaly = mean anomaly so we can use eq from slide 16 of the RV slides
def meanAnomaly(period, t, t_0): 
    """ 
    period: days
    t: days (time we are evaluating at)
    t_0: days (transit time OF THE PLANET)
    """
    nu = (2*np.pi/period) * (t - t_0) 

    if (nu.cgs.unit != u.dimensionless_unscaled):
        print("Units error")
        return None
    return nu * u.rad

In [119]:
# compute the anomalies of each planet at planet c's transit time
nu_K2b_T0c = meanAnomaly(P_K2b, T_0c, T_0b)
nu_K2c_T0c = meanAnomaly(P_K2c, T_0c, T_0c)
nu_K2d_T0c = meanAnomaly(P_K2c, T_0c, T_0d)

In [120]:
print(f"True anomaly of K2-3b at T_0c: {nu_K2b_T0c:.3f}")
print(f"True anomaly of K2-3c at T_0c: {nu_K2c_T0c:.3f}")
print(f"True anomaly of K2-3d at T_0c: {nu_K2d_T0c:.3f}")

True anomaly of K2-3b at T_0c: -0.713 rad
True anomaly of K2-3c at T_0c: 0.000 rad
True anomaly of K2-3d at T_0c: -3.555 rad


In [125]:
# RV of the star due to a planet is given by eq 2.29 from the RV slides 
# N.B. I ignored the long-term linear drift term since we are told to ignore COM motion and Earth motion so I take it that eliminates this drift as well as other systemics
def starRVfromPlanetatTime(K, w, nu, v_sys, e):
    """ 
    K: RV semi-amplitude (m/s)
    w: rad (arg of pericenter)
    nu: rad (anomaly EVALUATED AT TIME t) 
    v_sys: m/s  (systemic velocity)
    e: unitless (eccentricity)
    """

    RV = K * ( np.cos(np.deg2rad(w) + nu.value + e*np.cos(np.deg2rad(w))) ) + v_sys  

    if (RV.cgs.unit != (u.cm/u.s)):
        print("Units error")
        return None

    return RV.to(u.m/u.s)

In [126]:
# Compute the starRV due to each planet

starRV_K2b_T0c = starRVfromPlanetatTime(K_K2b, w, nu_K2b_T0c, v_sys, e)
starRV_K2c_T0c = starRVfromPlanetatTime(K_K2c, w, nu_K2c_T0c, v_sys, e)
starRV_K2d_T0c = starRVfromPlanetatTime(K_K2d, w, nu_K2d_T0c, v_sys, e)

In [127]:
print(f"RV of star due to K2-3b at T_0c: {starRV_K2b_T0c:.3f}")
print(f"RV of star due to K2-3c at T_0c: {starRV_K2c_T0c:.3f}")
print(f"RV of star due to K2-3d at T_0c: {starRV_K2d_T0c:.3f}")

RV of star due to K2-3b at T_0c: 1.485 m / s
RV of star due to K2-3c at T_0c: 0.000 m / s
RV of star due to K2-3d at T_0c: -0.404 m / s


In [129]:
# Add all the contributions from each planet to obtain the total star RV
starRV_total_T0c = starRV_K2b_T0c + starRV_K2c_T0c + starRV_K2d_T0c

print(f"Total RV of star due to K2-3b,c,d at T_0c: {starRV_total_T0c:.3f}")

Total RV of star due to K2-3b,c,d at T_0c: 1.081 m / s
